# 🔥 Feux de forêt — WildFire (CLIMADA)

Exploration du module `WildFire` de CLIMADA pour modéliser le risque incendie à partir de données satellites NASA FIRMS.

**Aléa** : Feux de forêt (WF)  
**Zone** : Méditerranée (sud France, Grèce, Portugal)  
**Données** : NASA FIRMS (MODIS/VIIRS) / synthétique  
**Métriques** : FRP (Fire Radiative Power), EAI, clustering spatial

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from climada.hazard import Hazard, Centroids
from climada.entity import Exposures, ImpactFuncSet, ImpactFunc
from climada.engine import Impact

try:
    from climada.hazard import WildFire
    from climada.entity.impact_funcs.wildfire import ImpfWildfire
    PETALS_OK = True
except ImportError:
    PETALS_OK = False
    print("⚠️ climada_petals non disponible — on utilisera des données synthétiques")

print(f"CLIMADA petals disponible : {PETALS_OK}")

## 1. Aléa — Feux de forêt (NASA FIRMS)

Le module `WildFire` de CLIMADA utilise les détections satellite **FIRMS** (Fire Information for Resource Management System) de la NASA :
- **MODIS** : résolution ~1 km, depuis 2000
- **VIIRS** : résolution ~375 m, depuis 2012

La variable d'intensité est le **FRP** (Fire Radiative Power, en MW) — proxy de l'intensité du feu.

### Téléchargement FIRMS
Les données FIRMS sont accessibles via l'API NASA : https://firms.modaps.eosdis.nasa.gov/

In [ ]:
# --- Téléchargement FIRMS réel (nécessite clé NASA) ---
# from climada.hazard import WildFire
# 
# # Depuis un DataFrame FIRMS (CSV téléchargé)
# df_firms = pd.read_csv('MODIS_C6_1_Europe_7d.csv')
# wildfire = WildFire.from_hist_fire_FIRMS(
#     df_firms=df_firms,
#     centr_res_factor=1.0,
#     description='Mediterranean wildfires'
# )
#
# # Par saisons de feu
# wildfire_seasons = WildFire.from_hist_fire_seasons_FIRMS(
#     df_firms=df_firms,
#     hemisphere='north',
#     year_start=2010,
#     year_end=2023
# )

# --- Données synthétiques ---
from scipy.sparse import csr_matrix, vstack

np.random.seed(42)

# Grille Méditerranée : 35-45°N, -10 à 30°E
lon = np.arange(-10, 30.5, 0.25)
lat = np.arange(35, 45.5, 0.25)
lon_grid, lat_grid = np.meshgrid(lon, lat)
lon_flat = lon_grid.flatten()
lat_flat = lat_grid.flatten()
n_centroids = len(lon_flat)

centroids = Centroids.from_lat_lon(lat_flat, lon_flat)
print(f"Centroids : {centroids.size} points sur la Méditerranée")

# Zones à risque élevé (forêts méditerranéennes)
fire_prone_zones = [
    {'name': 'Provence', 'lon': 5.5, 'lat': 43.5, 'radius': 1.5},
    {'name': 'Corse', 'lon': 9.0, 'lat': 42.0, 'radius': 0.8},
    {'name': 'Portugal_S', 'lon': -8.0, 'lat': 38.5, 'radius': 1.5},
    {'name': 'Grèce_Attique', 'lon': 23.7, 'lat': 38.0, 'radius': 1.0},
    {'name': 'Sardaigne', 'lon': 9.0, 'lat': 39.5, 'radius': 0.8},
    {'name': 'Catalogne', 'lon': 2.0, 'lat': 41.5, 'radius': 1.0},
    {'name': 'Andalousie', 'lon': -4.5, 'lat': 37.0, 'radius': 1.2},
    {'name': 'Sicile', 'lon': 14.5, 'lat': 37.5, 'radius': 0.8},
]

# Générer 20 saisons de feu (2004-2023)
n_seasons = 20
frequency = np.ones(n_seasons) / n_seasons

intensity_list = []
for s in range(n_seasons):
    season_intensity = np.zeros(n_centroids)
    
    # 3-8 feux majeurs par saison
    n_fires = np.random.randint(3, 9)
    for _ in range(n_fires):
        zone = fire_prone_zones[np.random.randint(len(fire_prone_zones))]
        # Position exacte avec perturbation
        fire_lon = zone['lon'] + np.random.normal(0, zone['radius'] * 0.5)
        fire_lat = zone['lat'] + np.random.normal(0, zone['radius'] * 0.3)
        
        dist = np.sqrt((lon_flat - fire_lon)**2 + (lat_flat - fire_lat)**2)
        
        # FRP : décroissance rapide, typiquement 50-500 MW pour les grands feux
        frp_max = np.random.exponential(200)  # MW
        fire_radius = np.random.uniform(0.1, 0.5)  # degrés
        frp = frp_max * np.exp(-0.5 * (dist / fire_radius)**2)
        
        season_intensity = np.maximum(season_intensity, frp)
    
    # Seuil de détection MODIS : ~10 MW
    season_intensity[season_intensity < 10] = 0
    intensity_list.append(csr_matrix(season_intensity))

intensity_matrix = vstack(intensity_list)

fire_haz = Hazard(
    haz_type='WF',
    centroids=centroids,
    intensity=intensity_matrix,
    frequency=frequency,
    event_id=np.arange(1, n_seasons + 1),
    event_name=[f'FireSeason_{2004 + i}' for i in range(n_seasons)],
    date=pd.date_range('2004-06-15', periods=n_seasons, freq='365D').to_numpy().astype('datetime64[ns]').astype(int) // 10**9,
    units='MW'
)

print(f"Hazard : {fire_haz.size} saisons de feu")
print(f"FRP max : {fire_haz.intensity.max():.1f} MW")
print(f"Cellules touchées (total) : {fire_haz.intensity.nnz}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Saison la plus intense
max_idx = np.array(fire_haz.intensity.max(axis=1).todense()).flatten().argmax()
max_frp = fire_haz.intensity[max_idx].toarray().flatten()

sc1 = axes[0].scatter(lon_flat[max_frp > 0], lat_flat[max_frp > 0], 
                       c=max_frp[max_frp > 0], s=8, cmap='hot_r', vmin=0)
axes[0].scatter(lon_flat[max_frp == 0], lat_flat[max_frp == 0], c='lightgray', s=0.5, alpha=0.3)
axes[0].set_title(f'Saison la plus intense : {fire_haz.event_name[max_idx]}')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
plt.colorbar(sc1, ax=axes[0], label='FRP (MW)')

# Fréquence de feu (nombre de saisons avec feu par cellule)
fire_count = np.array((fire_haz.intensity > 0).sum(axis=0)).flatten()
sc2 = axes[1].scatter(lon_flat, lat_flat, c=fire_count, s=2, cmap='YlOrRd', vmin=0)
axes[1].set_title('Fréquence de feu (nb saisons touchées)')
axes[1].set_xlabel('Longitude')
plt.colorbar(sc2, ax=axes[1], label='Nb saisons')

plt.tight_layout()
plt.show()

## 2. Clustering spatial et temporel

Analyse de la distribution spatio-temporelle des feux : identification des hotspots et tendance temporelle.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tendance temporelle : FRP total par saison
frp_per_season = np.array(fire_haz.intensity.sum(axis=1)).flatten()
years = np.arange(2004, 2004 + n_seasons)

axes[0].bar(years, frp_per_season, color='orangered', alpha=0.8)
z = np.polyfit(years, frp_per_season, 1)
axes[0].plot(years, np.polyval(z, years), 'k--', linewidth=2, label=f'Tendance : {z[0]:+.1f} MW/an')
axes[0].set_xlabel('Année')
axes[0].set_ylabel('FRP total (MW)')
axes[0].set_title('Intensité cumulée par saison de feu')
axes[0].legend()

# Nombre de cellules touchées par saison
cells_per_season = np.array((fire_haz.intensity > 0).sum(axis=1)).flatten()
axes[1].bar(years, cells_per_season, color='darkred', alpha=0.8)
z2 = np.polyfit(years, cells_per_season, 1)
axes[1].plot(years, np.polyval(z2, years), 'k--', linewidth=2, label=f'Tendance : {z2[0]:+.1f} cells/an')
axes[1].set_xlabel('Année')
axes[1].set_ylabel('Nb cellules touchées')
axes[1].set_title('Surface touchée par saison')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Exposition — Actifs méditerranéens

Portefeuille synthétique d'actifs dans les zones à risque incendie du bassin méditerranéen.

In [ ]:
from shapely.geometry import Point
import geopandas as gpd

n_assets = 400
np.random.seed(456)

med_cities = {
    'Marseille': (5.37, 43.30, 0.15),
    'Athènes': (23.73, 37.98, 0.15),
    'Lisbonne': (-9.14, 38.74, 0.12),
    'Barcelone': (2.17, 41.39, 0.12),
    'Rome': (12.50, 41.90, 0.10),
    'Montpellier': (3.88, 43.61, 0.08),
    'Nice': (7.26, 43.71, 0.08),
    'Ajaccio': (8.74, 41.93, 0.05),
    'Faro': (-7.93, 37.02, 0.05),
    'Palerme': (13.36, 38.12, 0.05),
    'Séville': (-5.98, 37.39, 0.05),
}

lons, lats, values = [], [], []
for city, (clon, clat, weight) in med_cities.items():
    n = int(n_assets * weight)
    lons.extend(np.random.normal(clon, 0.3, n))
    lats.extend(np.random.normal(clat, 0.2, n))
    values.extend(np.random.exponential(8e6, n))

remaining = n_assets - len(lons)
if remaining > 0:
    lons.extend(np.random.uniform(-8, 25, remaining))
    lats.extend(np.random.uniform(36, 44, remaining))
    values.extend(np.random.exponential(5e6, remaining))

gdf = gpd.GeoDataFrame({
    'value': np.array(values[:n_assets]),
    'impf_WF': np.ones(n_assets, dtype=int),
    'geometry': [Point(lo, la) for lo, la in zip(lons[:n_assets], lats[:n_assets])]
}, crs='EPSG:4326')

exposure = Exposures(gdf)
exposure.check()

print(f"Exposition : {len(exposure.gdf)} actifs")
print(f"Valeur totale : {exposure.gdf['value'].sum():,.0f} USD")

## 4. Vulnérabilité — Fonction d'impact feux

CLIMADA propose `ImpfWildfire.from_default_FIRMS()` avec une **sigmoïde** calibrée sur le FRP :

$$MDR(I) = \frac{1}{1 + \left(\frac{I_{half}}{I}\right)^{n}}$$

Où `I_half` (~295 MW) est le FRP à 50% de dommage.

In [ ]:
# --- CLIMADA default (si petals) ---
if PETALS_OK:
    try:
        impf_default = ImpfWildfire.from_default_FIRMS(i_half=295.01)
        print("Fonction par défaut FIRMS chargée")
    except Exception as e:
        print(f"Erreur : {e}")

# --- Fonctions custom ---
intensity_range = np.arange(0, 800, 1.0)

# Sigmoïde FIRMS (approximation du défaut CLIMADA)
i_half_default = 295.01
n_exp = 2.5
mdr_default = 1 / (1 + (i_half_default / np.maximum(intensity_range, 1e-6))**n_exp)
mdr_default[intensity_range < 10] = 0

# Sigmoïde plus conservative (seuil plus haut)
i_half_conserv = 400
mdr_conserv = 1 / (1 + (i_half_conserv / np.maximum(intensity_range, 1e-6))**n_exp)
mdr_conserv[intensity_range < 10] = 0

# Créer les fonctions
impf_firms = ImpactFunc(
    id=1, haz_type='WF',
    intensity=intensity_range,
    mdd=mdr_default,
    paa=np.ones_like(mdr_default),
    intensity_unit='MW',
    name='FIRMS default (i_half=295)'
)

impf_conservative = ImpactFunc(
    id=2, haz_type='WF',
    intensity=intensity_range,
    mdd=mdr_conserv,
    paa=np.ones_like(mdr_conserv),
    intensity_unit='MW',
    name='Conservative (i_half=400)'
)

# Visualisation
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(intensity_range, mdr_default, 'r-', linewidth=2, label=f'FIRMS default (i_half={i_half_default})')
ax.plot(intensity_range, mdr_conserv, 'b--', linewidth=2, label=f'Conservative (i_half={i_half_conserv})')
ax.axvline(x=i_half_default, color='r', linestyle=':', alpha=0.4)
ax.axvline(x=i_half_conserv, color='b', linestyle=':', alpha=0.4)
ax.set_xlabel('FRP (MW)')
ax.set_ylabel('MDR (Mean Damage Ratio)')
ax.set_title('Fonctions de vulnérabilité — Feux de forêt')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Impact — Calcul des pertes

In [ ]:
impf_use = ImpactFunc(
    id=1, haz_type='WF',
    intensity=impf_firms.intensity, mdd=impf_firms.mdd, paa=impf_firms.paa,
    intensity_unit='MW', name='FIRMS default'
)
impf_set = ImpactFuncSet([impf_use])

imp = Impact()
imp.calc(exposure, impf_set, fire_haz, save_mat=True)

print(f"EAI (Expected Annual Impact) : {imp.aai_agg:,.0f} USD")
print(f"Perte max événement :           {imp.at_event.max():,.0f} USD")
print(f"Nb saisons avec pertes :        {(imp.at_event > 0).sum()}/{n_seasons}")

# Pertes par saison
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(years, imp.at_event / 1e6, color='orangered', alpha=0.8)
ax.set_xlabel('Année')
ax.set_ylabel('Perte (M USD)')
ax.set_title('Pertes par saison de feu — Méditerranée')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Courbe de fréquence
fc = imp.calc_freq_curve()
axes[0].plot(fc.return_per, fc.impact / 1e6, 'r-', linewidth=2)
axes[0].set_xlabel('Période de retour (ans)')
axes[0].set_ylabel('Perte (M USD)')
axes[0].set_title('Courbe de fréquence — Feux')
axes[0].set_xscale('log')
axes[0].grid(True, alpha=0.3)

# EAI spatial
eai_exp = imp.eai_exp
gdf_plot = exposure.gdf.copy()
gdf_plot['eai'] = eai_exp

sc = axes[1].scatter(
    gdf_plot.geometry.x, gdf_plot.geometry.y,
    c=gdf_plot['eai'], s=15, cmap='YlOrRd', vmin=0
)
axes[1].set_title('EAI par actif — Feux de forêt')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
plt.colorbar(sc, ax=axes[1], label='EAI (USD/an)', shrink=0.7)

plt.tight_layout()
plt.show()

## 6. Synthèse

| Aspect | Détail |
|--------|--------|
| Source données | NASA FIRMS (MODIS C6.1 / VIIRS) |
| Variable | FRP — Fire Radiative Power (MW) |
| Résolution | ~1 km (MODIS), ~375 m (VIIRS) |
| Vulnérabilité | Sigmoïde calibrée (i_half ≈ 295 MW) |
| Couverture temporelle | 2000-présent (MODIS) |

### Points clés
- Le FRP est un bon proxy de l'intensité mais ne capture pas la durée d'exposition
- Les zones méditerranéennes montrent une forte variabilité inter-annuelle
- Le clustering spatial révèle les hotspots récurrents (Provence, Portugal sud, Grèce)
- Tendance à la hausse liée au changement climatique (sécheresses plus longues)

### Prochaines étapes
- Télécharger les données FIRMS réelles via l'API NASA
- Coupler avec les indices de sécheresse (cf. notebook 07_drought)
- Explorer la saisonnalité (mois de pic par sous-région)
- Projections futures : FWI (Fire Weather Index) sous CMIP6